In [1]:
import os
from pathlib import Path
import shutil

TEXBIN = "/Library/TeX/texbin"
if Path(TEXBIN).is_dir():
    os.environ["PATH"] = TEXBIN + ":" + os.environ.get("PATH", "")

print("pdflatex now:", shutil.which("pdflatex"))
print("latexmk now:", shutil.which("latexmk"))

pdflatex now: /Library/TeX/texbin/pdflatex
latexmk now: /Library/TeX/texbin/latexmk


### User input & data generation

In [2]:
# ===========================
# VERSION 1.0 — WORKING BASELINE
# Extended abstract generator
# Topic-agnostic variable handling
# LaTeX-safe postprocessing
# Date: 02/05/2026
# ===========================

In [3]:
from openai import OpenAI
import numpy as np
import pandas as pd
import re, json, os, shutil, subprocess, tempfile
from pathlib import Path
from dotenv import load_dotenv

# -----------------------------
# 0) Experiment configuration
# -----------------------------
load_dotenv()

experiment = {
    "context": (
        "Write an observations-style report of phosphate uptake kinetics in saline soils "
        "using time-series batch tests. Each record contains time, the run-specific "
        "equilibrium capacity, a kinetic rate constant, and the observed uptake at each time point. "
        "Describe what was done and what was recorded, not how data were generated."
    ),
    "motivation": (
        "Provide a clear lab-note summary: state N, recorded ranges and summary statistics, "
        "and describe how uptake changed with time across runs with different conditions."
    ),

    "NUM_DATASETS": 1,
    "N_OBS": 12,

    "inputs": {
        "t_(min)"    : ("min",   "uniform", [0.0, 120.0]),
        "Qe1_(mg/g)" : ("mg/g",  "uniform", [0.30, 2.50]),
        "k1_(1/min)" : ("1/min", "uniform", [0.002, 0.080]),
    },

    "outputs": {
        "qt_(mg/g)": (
            "mg/g",
            "Qe1_(mg/g) * (1.0 - np.exp(-k1_(1/min) * t_(min)))",
            ("gaussian", [0.0, 0.02])
        ),
    },

    "paper_words": 1000,
    "tok_paper": 2600,
    "model_id": "gpt-4.1",
    "out_dir": "Output/ExtendedAbstract_run",
}

experiment["constraints"] = (
    "[INSTRUCTIONS—DO NOT INCLUDE THIS PARAGRAPH IN THE PAPER]\n"
    "Write in PASSIVE VOICE ONLY.\n"
    "Do NOT use any sectioning commands (no \\section, \\subsection, etc.).\n"
    "The first paragraph must summarize the work WITHOUT using any variable names.\n"
    "From the second paragraph onward, variables may be used but must be italic and correctly subscripted.\n"
    "Do NOT include any equations or explicit functional forms.\n"
    "No bullet lists.\n"
    "Include the marker [[TABLE_HERE]] exactly once where the single data table should appear.\n"
    f"Use the actual sample size N={experiment['N_OBS']}.\n"
)

client = OpenAI()
rng = np.random.default_rng(2025)

OUT = Path(experiment["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) Utility: JSON robustness
# -----------------------------
def _extract_json_object(text: str) -> str | None:
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def _repair_json_loose(s: str) -> str:
    if not s:
        return s
    s = s.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    s = re.sub(r",\s*([}\]])", r"\1", s)

    out, in_str, esc = [], False, False
    for ch in s:
        if in_str:
            if esc:
                out.append(ch); esc = False
            else:
                if ch == "\\":
                    out.append(ch); esc = True
                elif ch == '"':
                    out.append(ch); in_str = False
                elif ch == "\n":
                    out.append("\\n")
                else:
                    out.append(ch)
        else:
            out.append(ch)
            if ch == '"':
                in_str = True
    return "".join(out)

def _json_from_llm(text: str) -> dict:
    block = _extract_json_object(text)
    if not block:
        return {}
    try:
        return json.loads(block)
    except json.JSONDecodeError:
        fixed = _repair_json_loose(block)
        try:
            return json.loads(fixed)
        except json.JSONDecodeError:
            return {}

# -----------------------------
# 2) Data generation (unchanged)
# -----------------------------
dists = {
    "uniform": lambda a, b, n: rng.uniform(a, b, n),
    "normal":  lambda m, s, n: rng.normal(m, s, n),
}

_safe_pat = re.compile(r"[^0-9a-zA-Z_]")
def _safe(col: str) -> str:
    return _safe_pat.sub("_", col)

def sample_inputs(n: int) -> pd.DataFrame:
    return pd.DataFrame({
        col: dists[dist](*params, n)
        for col, (_, dist, params) in experiment["inputs"].items()
    })

def add_outputs(df: pd.DataFrame) -> pd.DataFrame:
    for name, (_, formula, noise) in experiment["outputs"].items():
        expr = formula.replace("^", "**")
        env = {"np": np}
        for orig in df.columns:
            token = _safe(orig)
            env[token] = df[orig]
            expr = expr.replace(orig, token)
        y = eval(expr, env)
        mu, sd = noise[1]
        y = y + rng.normal(mu, sd, len(df))
        df[name] = y
    return df.round(6)

def stats_json(df: pd.DataFrame) -> str:
    return json.dumps(
        df.describe().T[["mean", "std", "min", "max"]].round(4).to_dict(),
        indent=2
    )

# -----------------------------
# 3) Robust symbol + unit handling (topic-agnostic)
# -----------------------------
def parse_col_key(key: str):
    # "Qe1_(mg/g)" -> ("Qe1", "mg/g")
    m = re.match(r"^(.+?)_\((.+)\)$", key)
    if m:
        return m.group(1), m.group(2)
    return key, ""

def unit_to_si(unit: str) -> str:
    # minimal mapper; expand as needed
    unit = (unit or "").strip()
    if unit == "min":
        return r"\si{min}"
    if unit == "mg/g":
        return r"\si{mg\per g}"
    if unit == "1/min":
        return r"\si{1\per min}"
    # fallback: print as plain text
    return unit

def symbol_to_math(symbol: str) -> str:
    """
    Generic conversion:
      qt   -> $\mathit{q}_t$
      Qe1  -> $\mathit{Q}_{e1}$
      k1   -> $\mathit{k}_1$
      t    -> $\mathit{t}$
      Vout -> $\mathit{V}_{out}$ (letters after first become subscript)
      alpha2 -> $\mathit{alpha}_2$ (digits become subscript)
    """
    s = symbol.strip()

    # if already has underscore notation, normalize into base/sub
    if "_" in s:
        base, sub = s.split("_", 1)
        sub = sub.strip("{}")
        return rf"$\mathit{{{base}}}_{{{sub}}}$"

    # single-letter variable
    if re.fullmatch(r"[A-Za-z]", s):
        return rf"$\mathit{{{s}}}$"

    # letter + digits (k1, x2, t10)
    m = re.fullmatch(r"([A-Za-z]+)([0-9]+)", s)
    if m:
        base, sub = m.group(1), m.group(2)
        # if base is multi-letter, keep base in mathit and digits as subscript
        return rf"$\mathit{{{base}}}_{{{sub}}}$"

    # Uppercase letter followed by letters/digits (Qe1, Vout, Ceq)
    m = re.fullmatch(r"([A-Za-z])([A-Za-z0-9]+)", s)
    if m:
        base, sub = m.group(1), m.group(2)
        return rf"$\mathit{{{base}}}_{{{sub}}}$"

    # fallback: italicize whole symbol
    return rf"$\mathit{{{s}}}$"

def build_var_map(experiment: dict) -> dict:
    keys = list(experiment["inputs"].keys()) + list(experiment["outputs"].keys())
    varmap = {}
    for k in keys:
        sym, unit = parse_col_key(k)
        varmap[sym] = {
            "math": symbol_to_math(sym),
            "unit": unit_to_si(unit),
        }
    return varmap

VARMAP = build_var_map(experiment)
VARS = sorted(VARMAP.keys(), key=len, reverse=True)  # longest first

def normalize_broken_variable_spellings(text: str) -> str:
    """
    Fix model outputs like:
      Q e,1 -> Qe1
      k 1   -> k1
      q t   -> qt
      mathitt -> t (or will be handled by replacement)
    """
    # collapse spaced forms: "q t" -> "qt", "k 1" -> "k1"
    text = re.sub(r"\b([A-Za-z])\s+([A-Za-z0-9])\b", r"\1\2", text)

    # specifically remove comma in patterns like "Q e,1" -> "Qe1"
    text = re.sub(r"\b([A-Za-z])\s*e\s*,\s*([0-9]+)\b", r"\1e\2", text)

    # remove comma inside subscripts that show up as "e,1"
    text = text.replace("e,1", "e1")

    return text


_MATH_INLINE_RE = re.compile(r"(\$.*?\$)", re.S)  # non-greedy, includes newlines


def apply_variable_math(text: str) -> str:
    """
    Replace variable mentions with VARMAP[sym]['math'] BUT NEVER inside existing $...$ math.
    This prevents nested math like: $\mathit{$\mathit{t}$}$.
    """
    if not text:
        return text

    text = normalize_broken_variable_spellings(text)

    # 1) convert \emph{sym} -> math form (safe: function replacement)
    def repl_emph(m):
        inner = re.sub(r"\s+", "", m.group(1)).strip("{}")
        if inner in VARMAP:
            return VARMAP[inner]["math"]
        return r"\emph{" + m.group(1) + "}"

    text = re.sub(r"\\emph\{([^}]*)\}", repl_emph, text)

    # 2) Replace bare symbols, but ONLY outside $...$ math blocks
    parts = _MATH_INLINE_RE.split(text)  # keeps delimiters ($...$) as separate elements
    for i, chunk in enumerate(parts):
        if not chunk:
            continue
        if chunk.startswith("$") and chunk.endswith("$"):
            # already math → do NOT touch
            continue

        # text chunk → safe to replace
        for sym in VARS:
            rep = VARMAP[sym]["math"]
            parts[i] = re.sub(
                rf"\b{re.escape(sym)}\b",
                lambda m, r=rep: r,
                parts[i])

    text = "".join(parts)

    # 3) escape stray underscores in remaining text (failsafe)
    text = re.sub(r"(?<!\\)_", r"\\_", text)

    # 4) fix common literal "mathit..." artifacts using lambda replacements
    rep_t  = VARMAP.get("t",  {"math": r"$\mathit{t}$"})["math"]
    rep_k1 = VARMAP.get("k1", {"math": r"$\mathit{k}_1$"})["math"]
    rep_qt = VARMAP.get("qt", {"math": r"$\mathit{q}_t$"})["math"]

    text = re.sub(r"\bmathitt\b",   lambda m, r=rep_t: r, text)
    text = re.sub(r"\bmathitk1\b",  lambda m, r=rep_k1: r, text)
    text = re.sub(r"\bmathitqt\b",  lambda m, r=rep_qt: r, text)
    text = re.sub(r"\$\\mathit\{\$(.*?)\$\}\$", r"$\1$", text)

    return text

# -----------------------------
# 4) First paragraph variable removal (unchanged idea)
# -----------------------------
def remove_variables_from_first_paragraph(body: str) -> str:
    parts = re.split(r"\n\s*\n", body, maxsplit=1)
    first = parts[0]
    rest  = parts[1] if len(parts) > 1 else ""

    # remove variable symbols broadly using VARS
    for sym in VARS:
        first = re.sub(rf"\b{re.escape(sym)}\b", "the measured quantity", first)
        first = re.sub(rf"\\emph\{{\s*{re.escape(sym)}\s*\}}", "the measured quantity", first)

    return first + ("\n\n" + rest if rest else "")

# -----------------------------
# 5) LLM calls
# -----------------------------
def llm_extended_body(stats: str) -> str:
    prompt = (
        f"{experiment['context']}\n\n"
        f"Motivation: {experiment['motivation']}\n\n"
        f"{experiment['constraints']}\n\n"
        f"Dataset summary statistics (JSON):\n{stats}\n\n"
        f"Write an ~{experiment['paper_words']}-word LaTeX write-up as connected paragraphs.\n"
        "Do NOT include any abstract environment or section headings.\n"
        "You MUST include [[TABLE_HERE]] exactly once where the single data table should appear.\n"
        "Do NOT include equations.\n"
        "Use \\emph{} for variables when they are used (but only from paragraph 2 onward).\n"
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=experiment["tok_paper"],
        temperature=0.4,
    )
    return res.choices[0].message.content.strip()

def llm_title(body_text: str) -> str:
    prompt = (
        "Return a single, concise plain-text title (8–14 words). "
        "No LaTeX commands, no quotes, no math.\n\n"
        "Body:\n" + body_text
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role":"user","content":prompt}],
        max_tokens=80,
        temperature=0.5,
    )
    title = res.choices[0].message.content.strip()
    title = re.sub(r"(?<!\\)%", r"\\%", title)
    title = title.replace("\n", " ")
    title = re.sub(r"\s+", " ", title).strip()
    return title if title else "Untitled"

def llm_table_caption(context, motivation, cols):
    prompt = (
        "Return JSON ONLY with key 'caption'.\n"
        "Write a SHORT caption (10–16 words). ONE sentence only WITHOUT active verbs.\n"
        "Do NOT include variable lists or ranges.\n"
        "Do NOT say generated/simulated/synthetic.\n\n"
        f"Context:\n{context}\n\n"
        f"Columns:\n{json.dumps(cols)}\n\n"
        f"N = {experiment['N_OBS']}\n"
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role":"user","content":prompt}],
        max_tokens=120,
        temperature=0.2,
    )
    data = _json_from_llm(res.choices[0].message.content)
    cap = (data.get("caption") or "").strip()
    if not cap:
        cap = f"Summary statistics for recorded batch tests (N = {experiment['N_OBS']})."
    cap = cap.replace("_", "")
    cap = re.sub(r"\s+", " ", cap).strip()
    if not cap.endswith("."):
        cap += "."
    return cap

# -----------------------------
# 6) Table generation (generic)
# -----------------------------
def pretty_col_tex_from_key(key: str) -> str:
    sym, unit = parse_col_key(key)
    v = VARMAP.get(sym, {}).get("math", sym)
    u = VARMAP.get(sym, {}).get("unit", unit)
    return f"{v} {u}".strip()

def df_to_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    desc = df.describe().loc[["mean", "std", "min", "max"]].copy()
    desc.index = ["Mean", "SD", "Min", "Max"]

    cols = list(desc.columns)
    header = " & ".join(pretty_col_tex_from_key(c) for c in cols)

    lines = [
        r"\begin{table}[H]",
        r"\centering",
        r"\caption{" + caption + r"}",
        r"\label{" + label + r"}",
        r"\setlength{\tabcolsep}{6pt}",
        r"\renewcommand{\arraystretch}{1.1}",
        r"\begin{tabular}{l" + "c"*len(cols) + r"}",
        r"\hline",
        r" & " + header + r" \\",
        r"\hline",
    ]

    for idx, row in desc.iterrows():
        vals = " & ".join(f"{float(v):.2f}" for v in row.values)
        lines.append(f"{idx} & {vals} \\\\")

    lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

def inject_table(body: str, table_tex: str) -> str:
    if "[[TABLE_HERE]]" in body:
        return body.replace("[[TABLE_HERE]]", table_tex, 1)
    parts = re.split(r"\n\s*\n", body, maxsplit=1)
    if len(parts) == 2:
        return parts[0] + "\n\n" + table_tex + "\n\n" + parts[1]
    return body + "\n\n" + table_tex

# -----------------------------
# 7) LaTeX document builder + compile
# -----------------------------
def build_tex_doc(title: str, body: str) -> str:
    return r"""\documentclass[10pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage{booktabs}
\usepackage{siunitx}
\usepackage{float}
\usepackage{microtype}
\sisetup{detect-weight=true, detect-family=true}
\setlength{\parskip}{6pt}
\setlength{\parindent}{0pt}

\title{""" + title + r"""}
\date{}

\begin{document}
\maketitle

\begin{center}
\textbf{Extended Abstract}
\end{center}
\vspace{-0.25em}

""" + body + r"""

\end{document}
"""

def ensure_tex_on_path():
    candidates = ["/Library/TeX/texbin", "/usr/texbin", "/usr/local/bin", "/usr/bin"]
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    for p in candidates:
        if p and os.path.isdir(p) and p not in path_parts:
            path_parts.insert(0, p)
    os.environ["PATH"] = os.pathsep.join(path_parts)

def compile_tex(tex_path: Path, workdir: Path):
    ensure_tex_on_path()
    proc = subprocess.run(
        ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", tex_path.name],
        cwd=workdir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    return proc.returncode == 0, proc.stdout




def show_first_latex_error(log_text: str, max_lines: int = 60) -> str:
    """
    Extracts the first LaTeX error block starting with '! ' and prints a short snippet.
    """
    lines = (log_text or "").splitlines()
    for i, ln in enumerate(lines):
        if ln.startswith("!"):
            start = max(0, i - 5)
            end = min(len(lines), i + max_lines)
            snippet = "\n".join(lines[start:end])
            return snippet
    return "\n".join(lines[:max_lines])





# -----------------------------
# 8) Main loop (NOW SAVES .tex FOR DEBUGGING)
# -----------------------------
for i in range(1, experiment["NUM_DATASETS"] + 1):
    print(f"▶ Dataset {i}/{experiment['NUM_DATASETS']}")

    df = add_outputs(sample_inputs(experiment["N_OBS"]))
    stats = stats_json(df)

    body_raw = llm_extended_body(stats)
    body_raw = remove_variables_from_first_paragraph(body_raw)

    cap = llm_table_caption(experiment["context"], experiment["motivation"],
                            [parse_col_key(c)[0] for c in list(df.columns)])
    table_tex = df_to_latex_table(df, cap, f"tab:data{i:02d}")

    body = inject_table(body_raw, table_tex)

    # Robust, topic-agnostic variable formatting
    body = apply_variable_math(body)

    title = llm_title(body)

    tex_doc = build_tex_doc(title, body)

    # Save TEX (debug)
    tex_out = OUT / f"paper{i:02d}.tex"
    tex_out.write_text(tex_doc, encoding="utf-8")

    # Compile in OUT so .log sits next to .tex (easy debugging)
    ok, log = compile_tex(tex_out, OUT)
    if not ok:
        (OUT / f"paper{i:02d}.log.txt").write_text(log, encoding="utf-8")
        print(f"   ✖ compile failed; saved log to paper{i:02d}.log.txt")
        print("\n--- First LaTeX error snippet ---")
        print(show_first_latex_error(log))
        raise RuntimeError("pdflatex failed; inspect the snippet above and the saved log.")

    # Move pdf to consistent name (pdflatex outputs paperXX.pdf in OUT)
    pdf_path = OUT / f"paper{i:02d}.pdf"
    if not pdf_path.exists():
        raise RuntimeError("Compilation succeeded but PDF not found where expected.")

    print("   ✔ saved", pdf_path)
    print("   ✔ saved", tex_out)

print("🎉 Done. Files saved in:", OUT.resolve())

▶ Dataset 1/1
   ✔ saved Output/ExtendedAbstract_run/paper01.pdf
   ✔ saved Output/ExtendedAbstract_run/paper01.tex
🎉 Done. Files saved in: /Users/md_halim_mondol/Desktop/STEPS-LLM/steps-llm/Src/Output/ExtendedAbstract_run
